# Land Cover Segmentation with a U-Net (DeepGlobe 2018)

A U-Net built from scratch in PyTorch that labels every pixel of a satellite image as one of 7 land cover classes: water, forest, rangeland, agriculture, barren, urban and unknown.

**Test set results:** mean IoU of **0.647** across the six land cover classes (0.555 when the rare "unknown" class is included), test loss 0.469.

**Notebook outline**
1. Setup and data download
2. Preprocessing the masks
3. Dataset and data loaders
4. Building the U-Net
5. Training (mixed precision, gradient clipping, checkpointing)
6. Evaluation on the test set

The investigation into a NaN test loss bug, and how it was traced to float16 overflow, is in [`debugging/nan_investigation.ipynb`](debugging/nan_investigation.ipynb).

Runs on Google Colab with a GPU (trained on a T4).

## 1. Setup and data download

In [ ]:
# Libraries Colab doesn't come with
!pip install -q kaggle albumentations

import os
import numpy as np
import torch
import torch.nn as n
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

In [ ]:
# Downloads the DeepGlobe 2018 land cover dataset from Kaggle
# Needs a Kaggle API key at ~/.kaggle/kaggle.json (keep this file private, never commit it)
!kaggle datasets download -d geoap96/deepglobe2018-landcover-segmentation-traindataset
!unzip -q deepglobe2018-landcover-segmentation-traindataset.zip
!unzip -q data.zip

In [ ]:
train_image_dir = "/content/data/training_data/images"
train_mask_dir = "/content/data/training_data/masks"

test_image_dir = "/content/data/test_data/images"
test_mask_dir = "/content/data/test_data/masks"

print("Training images:", len(os.listdir(train_image_dir)))
print("Training masks:", len(os.listdir(train_mask_dir)))

# Each image "<id>_sat.jpg" has a matching mask "<id>_mask.png"
for image in sorted(os.listdir(train_image_dir))[:5]:
    mask = image.replace("_sat.jpg", "_mask.png")
    print(image, "->", mask)

## 2. Preprocessing the masks

The masks store each class as an RGB colour. The loss function needs a single class ID per pixel, so every mask is converted from colours to class IDs.

| Class | ID | RGB colour |
|---|---|---|
| Water | 0 | (0, 0, 255) |
| Forest land | 1 | (0, 255, 0) |
| Rangeland | 2 | (255, 0, 255) |
| Agriculture land | 3 | (255, 255, 0) |
| Barren land | 4 | (255, 255, 255) |
| Urban land | 5 | (0, 255, 255) |
| Unknown | 6 | (0, 0, 0) |

In [ ]:
def rgb_to_class(mask):
  class_mask = np.zeros(mask.shape[:2], dtype=np.int64)

  class_map = {
        (0, 0, 255): 0,      # water
        (0, 255, 0): 1,      # forest_land
        (255, 0, 255): 2,    # rangeland
        (255, 255, 0): 3,    # agriculture_land
        (255, 255, 255): 4,  # barren_land
        (0, 255, 255): 5,    # urban_land
        (0, 0, 0): 6         # unknown
  }

  # For each colour, np.all(..., axis=-1) checks all 3 channels of every pixel and gives a
  # True/False map of the pixels that match that colour. Every True pixel is then set to the class ID.
  for colour, class_id in class_map.items():
    matches = np.all(mask == colour, axis=-1)
    class_mask[matches] = class_id

  return class_mask

**Optimisation:** the masks are converted once, up front, instead of inside the dataset's `__getitem__` every time an image is loaded. In my timing notes this took an epoch from about 20 minutes down to about 2 minutes.

In [ ]:
processed_mask_dir = "processed_masks"
processed_test_mask_dir = "processed_test_masks"


def preprocess_masks(source_dir, save_dir):
  os.makedirs(save_dir, exist_ok=True)

  for mask_name in os.listdir(source_dir):
    # Load the original RGB mask and convert it to class IDs
    mask = np.array(Image.open(os.path.join(source_dir, mask_name)))
    mask = rgb_to_class(mask)

    # Saved as uint8, since the class IDs only go up to 6
    Image.fromarray(mask.astype(np.uint8)).save(os.path.join(save_dir, mask_name))


preprocess_masks(train_mask_dir, processed_mask_dir)
preprocess_masks(test_mask_dir, processed_test_mask_dir)

In [ ]:
test_mask = np.array(
    Image.open(
        os.path.join(
            processed_mask_dir,
            os.listdir(processed_mask_dir)[0]
        )
    )
)

print(test_mask.shape)
print(np.unique(test_mask))

(2448, 2448)
[0 2 3 4 5]


Instead of reading the dataset documentation, the colours in the masks can also be found directly. `reshape(-1, 3)` flattens a mask into a list of RGB values (e.g. 100×100×3 becomes 10,000×3), and `np.unique(..., axis=0)` keeps each unique row, i.e. each unique colour. It only checks a few masks here because the full set takes a while.

In [ ]:
def find_unique_colours(mask_dir, max_masks=20):
  all_colours = set()

  for mask_name in os.listdir(mask_dir)[:max_masks]:
    mask = np.array(Image.open(os.path.join(mask_dir, mask_name)))

    unique_colours = np.unique(mask.reshape(-1, 3), axis=0)

    for colour in unique_colours:
      all_colours.add(tuple(colour))

  return all_colours


colours = find_unique_colours(train_mask_dir)
print(colours)
print("Number of colours:", len(colours))

## 3. Dataset and data loaders

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Random flips give the model more variety to learn from. They're applied to the image and mask together
train_transforms = A.Compose([
    A.Resize(512, 512),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Normalize(),
    ToTensorV2()
])

test_transforms = A.Compose([
    A.Resize(512, 512),
    A.Normalize(),
    ToTensorV2()
])

In [ ]:
from torch.utils.data import Dataset


class DeepGlobeLandCover(Dataset):  # inheriting from Dataset lets the rest of PyTorch (like DataLoader) use it
  def __init__(self, image_dir, mask_dir, transforms=None):
    super().__init__()
    # Stored on self so every method of the dataset can use them
    self.image_dir = image_dir
    self.mask_dir = mask_dir
    self.transforms = transforms

    self.images = sorted(os.listdir(image_dir))

  def __len__(self):
    return len(self.images)

  def __getitem__(self, idx):
    image_name = self.images[idx]
    mask_name = image_name.replace("_sat.jpg", "_mask.png")

    image_path = os.path.join(self.image_dir, image_name)
    mask_path = os.path.join(self.mask_dir, mask_name)

    # The network needs a consistent number of colour channels, so every image is converted to RGB
    image = Image.open(image_path).convert("RGB")
    mask = Image.open(mask_path)

    # Albumentations works with NumPy arrays of shape (height, width, channels)
    image = np.array(image)
    mask = np.array(mask)

    if self.transforms:
      transformed = self.transforms(image=image, mask=mask)

      image = transformed["image"]
      mask = transformed["mask"].long()

    return image, mask

In [ ]:
from torch.utils.data import random_split

train_dataset = DeepGlobeLandCover(train_image_dir, processed_mask_dir, train_transforms)
test_dataset = DeepGlobeLandCover(test_image_dir, processed_test_mask_dir, test_transforms)

# Validation set: 20% of the training images are held back, so the model is checked on images it never trains on
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size

# Fixed seed so every Colab session makes the same split. Without it, resuming from a checkpoint would give a
# new validation set, mostly made of images the model had already trained on
train_dataset, val_dataset = random_split(
    train_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

print("Train:", len(train_dataset), "Validation:", len(val_dataset), "Test:", len(test_dataset))

In [ ]:
image, mask = train_dataset[0]

print(image.shape, image.dtype)
print(mask.shape, mask.dtype)  # CrossEntropyLoss expects class IDs as int64 (torch.long)
print(torch.unique(mask))

In [ ]:
import matplotlib.pyplot as plt

# A.Normalize() shifted the pixel values for training, so they're turned back into normal colours for display
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


def to_display_image(image):
  return (image.cpu() * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()


# The dataset's own colours for each class ID, so masks look like the originals
class_colours = np.array([
    [0, 0, 255], [0, 255, 0], [255, 0, 255], [255, 255, 0],
    [255, 255, 255], [0, 255, 255], [0, 0, 0]
], dtype=np.uint8)


def to_display_mask(mask):
  return class_colours[mask.cpu().numpy()]


plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.imshow(to_display_image(image))
plt.title("Satellite Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(to_display_mask(mask))
plt.title("Segmentation Mask")
plt.axis("off")

plt.show()

In [ ]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(train_dataset,
                              batch_size=8,
                              shuffle=True,
                              num_workers=0,
                              pin_memory=True,
                              persistent_workers=False)

val_dataloader = DataLoader(val_dataset,
                            batch_size=8,
                            shuffle=False,
                            num_workers=0,
                            pin_memory=True,
                            persistent_workers=False)

test_dataloader = DataLoader(test_dataset,
                             batch_size=4,
                             shuffle=False,
                             num_workers=0,
                             pin_memory=True,
                             persistent_workers=False)

In [ ]:
image, mask=next(iter(train_dataloader))

print(image.shape)
print(mask.shape)
print(image.dtype)
print(mask.dtype)

torch.Size([8, 3, 512, 512])
torch.Size([8, 512, 512])
torch.float32
torch.int64


## 4. Building the U-Net

A U-Net has two halves:
- **Encoder (down path):** each step halves the image size and doubles the number of feature maps, so the network learns *what* is in the image.
- **Decoder (up path):** each step doubles the image size back up. **Skip connections** join each decoder step with the encoder output of the same size, which brings back the fine detail of *where* things are.

It's built from three blocks: `DoubleConv`, `Down` and `Up`.

### Double convolution block

In [ ]:
class DoubleConv(n.Module):
  def __init__(self, in_channels, out_channels):
    super().__init__()
    self.conv = n.Sequential(
        n.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            padding=1,
            # No bias, because BatchNorm straight after subtracts the mean (cancelling any bias)
            # and then adds its own learnable shift
            bias=False
        ),
        # Normalises the feature maps produced by the convolution
        n.BatchNorm2d(out_channels),
        n.ReLU(inplace=True),

        n.Conv2d(out_channels,
                 out_channels,
                 kernel_size=3,
                 padding=1,
                 bias=False),
        n.BatchNorm2d(out_channels),
        n.ReLU(inplace=True)
    )

  def forward(self, x):
    return self.conv(x)

In [ ]:
# Shape check: padding=1 keeps the 512x512 size, and the channels go from 3 to 64

block = DoubleConv(3,64)

x= torch.randn(1,3,512,512)

output=block(x)

print(output.shape)

torch.Size([1, 64, 512, 512])


### Down block

**How the channels connect (my notes):** with 64 input feature maps and 128 output channels, every input map feeds into every output channel. Each output channel has its own 3×3 kernel for each of the 64 input maps, so its filter has shape 64×3×3 (576 learnable weights). The 64 results are added together to make that output channel's single feature map.

In [ ]:
class Down(n.Module):
  def __init__(self, in_channels, out_channels):
    super().__init__()
    self.down = n.Sequential(
        # Max pooling slides a 2x2 window over the feature map and keeps the largest value in it.
        # stride=2 moves the window 2 pixels at a time, so the height and width are halved
        n.MaxPool2d(kernel_size=2, stride=2),

        DoubleConv(in_channels, out_channels)
    )

  def forward(self, x):
    return self.down(x)

In [ ]:
# Shape check: size halves, channels double

down = Down(64,128)

x=torch.randn(1,64,512,512)

down_result= down(x)

down_result.shape

torch.Size([1, 128, 256, 256])

### Up block

In [ ]:
class Up(n.Module):
  def __init__(self, in_channels, out_channels):
    super().__init__()
    # ConvTranspose2d (upsampling): for each input value it multiplies the kernel's weights by that value
    # and writes the result into the output. The stride decides where the next result is written, and any
    # results that land on the same output position are added together.
    # in_channels // 2 is the number of output channels, so the channels halve as the size doubles
    self.up = n.ConvTranspose2d(
        in_channels,
        in_channels // 2,
        kernel_size=2,
        stride=2
    )

    # After joining with the skip connection the channel count is back to in_channels
    self.conv = DoubleConv(in_channels, out_channels)

  def forward(self, x1, x2):
    x1 = self.up(x1)

    # Skip connection: join the upsampled features with the encoder features of the same size
    x = torch.cat([x2, x1], dim=1)

    return self.conv(x)

### Full U-Net

The final 1×1 convolution looks at each pixel position across all 64 feature maps and turns it into 7 scores, one per class.

In [ ]:
class UNet(n.Module):
  def __init__(self, in_channels, num_classes):
    super().__init__()

    self.inc = DoubleConv(in_channels, 64)

    # Encoder
    self.down1 = Down(64, 128)
    self.down2 = Down(128, 256)
    self.down3 = Down(256, 512)
    self.down4 = Down(512, 1024)

    # Decoder
    self.up1 = Up(1024, 512)
    self.up2 = Up(512, 256)
    self.up3 = Up(256, 128)
    self.up4 = Up(128, 64)

    self.outc = n.Conv2d(64, num_classes, kernel_size=1)

  def forward(self, x):
    x1 = self.inc(x)
    x2 = self.down1(x1)
    x3 = self.down2(x2)
    x4 = self.down3(x3)
    x5 = self.down4(x4)

    # up1 and up2 run in float32 even during mixed precision training.
    # In float16 the largest number is 65,504. up1's first conv adds up 1024*3*3 = 9,216 inputs per output
    # pixel, so the sum overflowed to inf, which then corrupted BatchNorm's running stats (NaN test loss).
    # .float() converts the float16 encoder outputs; outside autocast it does nothing.
    # Full investigation: debugging/nan_investigation.ipynb
    with torch.autocast(device_type=x.device.type, enabled=False):
      x = self.up1(x5.float(), x4.float())
      x = self.up2(x, x3.float())

    x = self.up3(x, x2)
    x = self.up4(x, x1)

    return self.outc(x)

## 5. Training

In [ ]:
model = UNet(in_channels=3, num_classes=7).to(device)

# torch.compile speeds up training. Note: it stops forward hooks from firing, so debugging hooks
# need to go on the uncompiled model (model._orig_mod)
model = torch.compile(model)

In [ ]:
loss_fn = n.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.parameters(), lr=0.001)

# Mixed precision: most layers run in float16 for speed. GradScaler scales the loss up before
# backpropagation so small gradients don't round down to zero in float16
scaler = torch.amp.GradScaler("cuda")

In [ ]:
print("GPU AVAILABLE:", torch.cuda.is_available())
print("TORCH VR:", torch.__version__)
print("DEVICE:", device)

if torch.cuda.is_available():
  print("GPU NAME:", torch.cuda.get_device_name())

GPU AVAILABLE: True
TORCH VR: 2.11.0+cu128
DEVICE: cuda
GPU NAME: Tesla T4


Checkpoints are saved to Google Drive after every epoch, so training can continue across Colab sessions (Colab's own storage is wiped when the session ends).

In [ ]:
from google.colab import drive

# Mounts Google Drive inside Colab's temporary storage, so files saved there are kept permanently
drive.mount("/content/drive")

drive_checkpoint_dir = "/content/drive/MyDrive/deepglobe_checkpoints"
os.makedirs(drive_checkpoint_dir, exist_ok=True)

drive_checkpoint = os.path.join(drive_checkpoint_dir, "latest_checkpoint.pth")

# Also saved to Colab's temporary storage
local_checkpoint = "/content/latest_checkpoint.pth"

Mounted at /content/drive


In [ ]:
# Resume from the last checkpoint if there is one, otherwise start from epoch 0
start_epoch = 0

if os.path.exists(drive_checkpoint):
  checkpoint = torch.load(drive_checkpoint, map_location=device)

  state_dict = checkpoint["model_state_dict"]

  # torch.compile adds "_orig_mod." to the start of every weight name, so the names are matched up first
  if hasattr(model, "_orig_mod"):
    state_dict = {
        f"_orig_mod.{k}" if not k.startswith("_orig_mod.") else k: v
        for k, v in state_dict.items()
    }
  else:
    state_dict = {
        k.replace("_orig_mod.", ""): v
        for k, v in state_dict.items()
    }

  model.load_state_dict(state_dict)
  optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

  start_epoch = checkpoint["epoch"]
  train_loss_avg = checkpoint["train_loss"]
  val_loss_avg = checkpoint["val_loss"]

  print("Resuming from epoch:", start_epoch)
  print("Previous training loss:", train_loss_avg)
  print("Previous validation loss:", val_loss_avg)

Resuming from epoch: 100
Previous training loss: 0.4301866288634314
Previous validation loss: 0.43322862353589797


In [ ]:
import math
from tqdm.auto import tqdm

epochs = 100

for epoch in range(start_epoch, epochs):
  # ---- TRAIN ----
  model.train()
  total_train_loss = 0

  for images, mask in tqdm(train_dataloader, desc=f"Epoch | {epoch + 1}/{epochs}"):

    images = images.to(device, non_blocking=True)
    mask = mask.to(device, non_blocking=True)

    if not torch.isfinite(images).all():
      raise RuntimeError(f"🚨 Non-finite training image detected at epoch {epoch + 1}")

    optimizer.zero_grad()

    # Mixed precision: float16 where it's safe, which reduces computation and memory
    with torch.autocast(device_type="cuda"):
      outputs = model(images)
      loss = loss_fn(outputs, mask)

      if not torch.isfinite(loss).all():
        raise RuntimeError(f"NaN/Inf in the loss at epoch {epoch + 1}")

    total_train_loss += loss.item()

    # Produces scaled gradients
    scaler.scale(loss).backward()

    # Unscale first so gradient clipping below works on the real gradients, not the scaled ones.
    # scaler.step() knows they're already unscaled, so they don't get unscaled twice
    scaler.unscale_(optimizer)

    # With float16 + GradScaler an occasional inf/NaN gradient is normal: scaler.step() skips that
    # update and scaler.update() lowers the scale, so it's reported rather than stopping training
    for name, p in model.named_parameters():
      if p.grad is not None and not torch.isfinite(p.grad).all():
        print(f"⚠️ Non-finite gradient in {name}, GradScaler will skip this step")
        break

    # Gradient clipping: caps the size of the gradients so one bad batch can't make a huge update
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

    if not torch.isfinite(grad_norm):
      print(f"⚠️ Non-finite gradient norm at epoch {epoch + 1}, step skipped")

    # Applies the gradients to the weights (skipped if they contain inf/NaN)
    scaler.step(optimizer)

    # Updates how much the loss will be scaled by next time
    scaler.update()

    for name, p in model.named_parameters():
      if not torch.isfinite(p).all():
        raise RuntimeError(f"🚨 Non-finite parameter detected {name}")

  train_loss_avg = total_train_loss / len(train_dataloader)

  # ---- VALIDATE ----
  model.eval()
  val_loss_total = 0

  with torch.inference_mode():
    for image, mask in tqdm(val_dataloader, desc="Validation"):

      image = image.to(device, non_blocking=True)
      mask = mask.to(device, non_blocking=True)

      if not torch.isfinite(image).all():
        raise RuntimeError(f"🚨 Non-finite validation image detected at epoch {epoch + 1}")

      with torch.autocast(device_type="cuda"):
        val_output = model(image)
        val_loss = loss_fn(val_output, mask)

      val_loss_total += val_loss.item()

  val_loss_avg = val_loss_total / len(val_dataloader)

  print(f"Epoch | {epoch + 1}, train_loss | {train_loss_avg:.4}, val_loss_avg | {val_loss_avg:.4}")

  # Never overwrite a good checkpoint with a broken model: if the val loss is NaN/inf, stop instead of saving
  if not math.isfinite(val_loss_avg):
    raise RuntimeError(f"🚨 Non-finite val loss at epoch {epoch + 1}, checkpoint NOT saved")

  checkpoint = {
      "epoch": epoch + 1,
      "model_state_dict": model.state_dict(),
      "optimizer_state_dict": optimizer.state_dict(),
      "train_loss": train_loss_avg,
      "val_loss": val_loss_avg
  }

  torch.save(checkpoint, local_checkpoint)
  torch.save(checkpoint, drive_checkpoint)  # permanent copy on Drive

## 6. Evaluation on the test set

**IoU (Intersection over Union)** is the standard score for segmentation. For each class it compares the pixels the model predicted as that class with the pixels that really are that class:

IoU = TP / (TP + FP + FN)

- **TP** (true positive): predicted the class, and it's correct
- **FP** (false positive): predicted the class, but it's something else
- **FN** (false negative): missed pixels that really are the class

In [ ]:
model.eval()

num_classes = 7

with torch.inference_mode():

  total_test_loss = 0

  total_TP = torch.zeros(num_classes)
  total_FP = torch.zeros(num_classes)
  total_FN = torch.zeros(num_classes)

  for image, mask in tqdm(test_dataloader, desc="Testing model"):

    image = image.to(device, non_blocking=True)
    mask = mask.to(device, non_blocking=True)

    test_output = model(image)

    # The predicted class for each pixel is the one with the highest score
    pred_mask = torch.argmax(test_output, dim=1)

    for class_id in range(num_classes):

      # True/False maps of where this class was predicted and where it really is
      pred_class = pred_mask == class_id
      true_class = mask == class_id

      TP = (pred_class & true_class).sum()    # predicted the class, and it's there
      FP = (pred_class & ~true_class).sum()   # predicted the class, but it isn't there
      FN = (~pred_class & true_class).sum()   # missed the class where it is there

      # Totals are kept across all test batches
      total_TP[class_id] += TP.item()
      total_FP[class_id] += FP.item()
      total_FN[class_id] += FN.item()

    test_loss = loss_fn(test_output, mask)
    total_test_loss += test_loss.item()

  test_loss_avg = total_test_loss / len(test_dataloader)

  # IoU per class (1e-6 avoids dividing by zero)
  iou = total_TP / (total_TP + total_FP + total_FN + 1e-6)

  print(f"test_loss: {test_loss_avg:.4}")

  print("IoU per class: ")
  for class_id in range(num_classes):
    print(f"Class: {class_id} : {iou[class_id]:.4f}")

Testing model:   0%|          | 0/30 [00:00<?, ?it/s]

test_loss: 0.469
IoU per class: 
Class: 0 : 0.6647
Class: 1 : 0.7572
Class: 2 : 0.2888
Class: 3 : 0.8330
Class: 4 : 0.6130
Class: 5 : 0.7250
Class: 6 : 0.0000


### Results

| Class | IoU |
|---|---|
| Water | 0.665 |
| Forest land | 0.757 |
| Rangeland | 0.289 |
| Agriculture land | 0.833 |
| Barren land | 0.613 |
| Urban land | 0.725 |
| Unknown | 0.000 |
| **Mean IoU (6 land cover classes)** | **0.647** |
| Mean IoU (all 7 classes) | 0.555 |

The "unknown" class is under 0.1% of all pixels, so the model never learns it. The DeepGlobe challenge itself scores mean IoU over the six real classes, without unknown.

Rangeland is the hardest class: it looks similar to agriculture and barren land from above, and it covers far fewer pixels than agriculture (see below).

### Example predictions

In [ ]:
# Shows test images next to their true masks and the model's predicted masks
images, masks = next(iter(test_dataloader))

model.eval()
with torch.inference_mode():
  predictions = torch.argmax(model(images.to(device)), dim=1).cpu()

rows = len(images)
plt.figure(figsize=(12, 4 * rows))

for i in range(rows):
  plt.subplot(rows, 3, 3 * i + 1)
  plt.imshow(to_display_image(images[i]))
  plt.title("Satellite Image")
  plt.axis("off")

  plt.subplot(rows, 3, 3 * i + 2)
  plt.imshow(to_display_mask(masks[i]))
  plt.title("True Mask")
  plt.axis("off")

  plt.subplot(rows, 3, 3 * i + 3)
  plt.imshow(to_display_mask(predictions[i]))
  plt.title("Predicted Mask")
  plt.axis("off")

plt.tight_layout()
plt.savefig("predictions.png", dpi=100)  # for the README
plt.show()

### Class balance

Counting how many training pixels belong to each class:

In [ ]:
import numpy as np
from PIL import Image
import os

class_counts = np.zeros(7, dtype=np.int64)

for filename in os.listdir(processed_mask_dir):

    mask = np.array(
        Image.open(
            os.path.join(processed_mask_dir, filename)
        )
    )

    for c in range(7):
        class_counts[c] += np.sum(mask == c)

print(class_counts)

[ 138399288  444916091  343123454 2379645852  323391391  461187501    2353255]


Agriculture land makes up about 58% of all training pixels, while water is about 3% and unknown is under 0.1%. This imbalance is a likely reason for the weaker scores on the smaller classes. A next step would be a class-weighted loss or Dice loss.

## Performance notes (from my timing logs)

- Converting the masks from RGB to class IDs up front, instead of inside `__getitem__`: an epoch went from about 20 minutes to about 2 minutes.
- Mixed precision (float16) training with `torch.autocast` and `GradScaler`.
- `torch.compile` on the model.